In [1]:
# ============================================================
# SEMANTIC THRESHOLD tau SENSITIVITY ANALYSIS
# ============================================================

import time
import itertools
import pandas as pd
import networkx as nx
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

# tau values to compare
TAU_VALUES = [0.30, 0.40, 0.50, 0.60]

# Evaluate the final backbone used in the paper
W_BACKBONE = 20

# ============================================================
# 1) NORMALIZE KEYWORDS ONCE
# ============================================================

docs_keywords = []

for kws in df["keywords_llm"].tolist():
    kws_clean = [
        k.strip().lower()
        for k in kws
        if isinstance(k, str) and k.strip()
    ]
    kws_clean = sorted(set(kws_clean))
    if len(kws_clean) > 0:
        docs_keywords.append(kws_clean)

print(f"Documents with valid keywords: {len(docs_keywords):,}")

# ============================================================
# 2) CREATE UNIQUE VOCABULARY AND EMBEDDINGS ONCE
# ============================================================

vocab = sorted(set(k for kws in docs_keywords for k in kws))

print(f"Unique vocabulary: {len(vocab):,}")

print("Computing global keyword embeddings...")
emb_matrix = model.encode(
    vocab,
    normalize_embeddings=True,
    show_progress_bar=True
)

embeddings_dict = {
    kw: emb_matrix[i]
    for i, kw in enumerate(vocab)
}

print("Embeddings ready.")

# ============================================================
# 3) FUNCTION TO BUILD THE CRS FOR A GIVEN tau
# ============================================================

def build_crs_for_tau(docs_keywords, embeddings_dict, tau):
    G = nx.Graph()

    for idx, kws in enumerate(docs_keywords, start=1):

        # Add nodes and document frequency
        for k in kws:
            if G.has_node(k):
                G.nodes[k]["doc_freq"] += 1
            else:
                G.add_node(k, doc_freq=1)

        if len(kws) < 2:
            continue

        # Local embedding matrix
        local_emb = np.array([embeddings_dict[k] for k in kws])
        S = cosine_similarity(local_emb)

        # Local edges filtered by tau
        for i, j in itertools.combinations(range(len(kws)), 2):
            sim = float(S[i, j])

            if sim >= tau:
                a, b = kws[i], kws[j]

                if G.has_edge(a, b):
                    G[a][b]["weight"] += 1
                    G[a][b]["sim_sum"] += sim
                    G[a][b]["sim_count"] += 1
                else:
                    G.add_edge(
                        a,
                        b,
                        weight=1,
                        sim_sum=sim,
                        sim_count=1
                    )

    # Mean similarity
    for u, v, d in G.edges(data=True):
        d["sim_mean"] = d["sim_sum"] / d["sim_count"]

    return G

# ============================================================
# 4) FUNCTION TO COMPUTE METRICS
# ============================================================

def graph_metrics(G, tau, w_backbone=20):
    n = G.number_of_nodes()
    e = G.number_of_edges()
    density = nx.density(G) if n > 1 else 0

    components = nx.number_connected_components(G) if n > 0 else 0

    if n > 0 and e > 0:
        lcc_nodes = max(nx.connected_components(G), key=len)
        L = G.subgraph(lcc_nodes).copy()
        lcc_nodes_n = L.number_of_nodes()
        lcc_edges_n = L.number_of_edges()
        lcc_ratio = lcc_nodes_n / n
    else:
        lcc_nodes_n = 0
        lcc_edges_n = 0
        lcc_ratio = 0

    # Backbone w >= 20
    H = nx.Graph()
    H.add_nodes_from(G.nodes(data=True))
    H.add_edges_from([
        (u, v, d)
        for u, v, d in G.edges(data=True)
        if float(d.get("weight", 1)) >= w_backbone
    ])
    H.remove_nodes_from([node for node in list(H.nodes()) if H.degree(node) == 0])

    hn = H.number_of_nodes()
    he = H.number_of_edges()
    h_density = nx.density(H) if hn > 1 else 0
    h_components = nx.number_connected_components(H) if hn > 0 else 0

    if hn > 0 and he > 0:
        h_lcc_nodes = max(nx.connected_components(H), key=len)
        HL = H.subgraph(h_lcc_nodes).copy()
        h_lcc_n = HL.number_of_nodes()
        h_lcc_e = HL.number_of_edges()
        h_lcc_ratio = h_lcc_n / hn
    else:
        h_lcc_n = 0
        h_lcc_e = 0
        h_lcc_ratio = 0

    # Louvain modularity in the backbone
    try:
        import community as community_louvain
        if hn > 0 and he > 0:
            part = community_louvain.best_partition(
                H,
                weight="weight",
                random_state=42
            )
            modularity = community_louvain.modularity(
                part,
                H,
                weight="weight"
            )
            n_communities = len(set(part.values()))
        else:
            modularity = np.nan
            n_communities = 0
    except Exception:
        modularity = np.nan
        n_communities = np.nan

    return {
        "tau": tau,
        "nodes_global": n,
        "edges_global": e,
        "density_global": density,
        "components_global": components,
        "lcc_nodes_global": lcc_nodes_n,
        "lcc_edges_global": lcc_edges_n,
        "lcc_ratio_global": lcc_ratio,
        "nodes_backbone_w20": hn,
        "edges_backbone_w20": he,
        "density_backbone_w20": h_density,
        "components_backbone_w20": h_components,
        "lcc_nodes_backbone_w20": h_lcc_n,
        "lcc_edges_backbone_w20": h_lcc_e,
        "lcc_ratio_backbone_w20": h_lcc_ratio,
        "modularity_backbone_w20": modularity,
        "communities_backbone_w20": n_communities
    }

# ============================================================
# 5) RUN SENSITIVITY ANALYSIS
# ============================================================

sensitivity_results = []

for tau in TAU_VALUES:
    print("\n" + "="*60)
    print(f"Building CRS for tau = {tau}")
    print("="*60)

    start = time.time()

    G_tau = build_crs_for_tau(
        docs_keywords=docs_keywords,
        embeddings_dict=embeddings_dict,
        tau=tau
    )

    metrics = graph_metrics(
        G=G_tau,
        tau=tau,
        w_backbone=W_BACKBONE
    )

    elapsed = time.time() - start
    metrics["runtime_seconds"] = elapsed

    sensitivity_results.append(metrics)

    print(f"tau = {tau}")
    print(f"Global |V| = {metrics['nodes_global']:,}")
    print(f"Global |E| = {metrics['edges_global']:,}")
    print(f"Global density = {metrics['density_global']:.6f}")
    print(f"Global LCC ratio = {metrics['lcc_ratio_global']:.4f}")
    print(f"Backbone w >= {W_BACKBONE} |V| = {metrics['nodes_backbone_w20']:,}")
    print(f"Backbone w >= {W_BACKBONE} |E| = {metrics['edges_backbone_w20']:,}")
    print(f"Backbone LCC ratio = {metrics['lcc_ratio_backbone_w20']:.4f}")
    print(f"Backbone modularity = {metrics['modularity_backbone_w20']}")
    print(f"Time = {elapsed/60:.2f} minutes")

# ============================================================
# 6) FINAL TABLE
# ============================================================

df_tau_sensitivity = pd.DataFrame(sensitivity_results)

pd.set_option("display.max_columns", None)
display(df_tau_sensitivity)

# Save results
OUT_TAU = "outputs/tau_sensitivity.csv"
df_tau_sensitivity.to_csv(OUT_TAU, index=False)

print(f"\nResults saved to:")
print(OUT_TAU)

# ============================================================
# 7) SUMMARY TABLE FOR THE PAPER
# ============================================================

summary_cols = [
    "tau",
    "nodes_global",
    "edges_global",
    "density_global",
    "lcc_ratio_global",
    "nodes_backbone_w20",
    "edges_backbone_w20",
    "lcc_ratio_backbone_w20",
    "modularity_backbone_w20",
    "communities_backbone_w20"
]

df_tau_summary = df_tau_sensitivity[summary_cols].copy()

display(df_tau_summary)

OUT_TAU_SUMMARY = "outputs/tau_sensitivity_summary.csv"
df_tau_summary.to_csv(OUT_TAU_SUMMARY, index=False)

print(f"\nSummary table saved to:")
print(OUT_TAU_SUMMARY)

Documents with valid keywords: 52,946
Unique vocabulary: 56,635
Computing global keyword embeddings...


Batches:   0%|          | 0/1770 [00:00<?, ?it/s]

Batches:   0%|          | 2/1770 [00:00<02:29, 11.86it/s]

Batches:   0%|          | 4/1770 [00:00<01:57, 15.01it/s]

Batches:   0%|          | 6/1770 [00:00<01:50, 15.98it/s]

...

tau = 0.6
Global |V| = 56,635
Global |E| = 28,118
Global density = 0.000018
Global LCC ratio = 0.2249
Backbone w >= 20 |V| = 153
Backbone w >= 20 |E| = 173
Backbone LCC ratio = 0.8693
Backbone modularity = 0.5964805004648758
Time = 0.17 minutes


,tau,nodes_global,edges_global,density_global,components_global,lcc_nodes_global,lcc_edges_global,lcc_ratio_global,nodes_backbone_w20,edges_backbone_w20,density_backbone_w20,components_backbone_w20,lcc_nodes_backbone_w20,lcc_edges_backbone_w20,lcc_ratio_backbone_w20,modularity_backbone_w20,communities_backbone_w20,runtime_seconds
0,0.3,56635,186795,0.000116,8695,46484,185044,0.820765,519,837,0.006227,1,519,837,1.000000,0.337002,6,11.955610
1,0.4,56635,109022,0.000068,19856,34316,106297,0.605915,408,608,0.007323,2,406,607,0.995098,0.356044,7,11.189187
2,0.5,56635,57928,0.000036,31193,22413,54690,0.395745,255,344,0.010622,8,232,328,0.909804,0.349522,10,11.022010
3,0.6,56635,28118,0.000018,40821,12735,24867,0.224861,153,173,0.014878,9,133,160,0.869281,0.596481,12,10.091630



Results saved to:
outputs/tau_sensitivity.csv


,tau,nodes_global,edges_global,density_global,lcc_ratio_global,nodes_backbone_w20,edges_backbone_w20,lcc_ratio_backbone_w20,modularity_backbone_w20,communities_backbone_w20
0,0.3,56635,186795,0.000116,0.820765,519,837,1.000000,0.337002,6
1,0.4,56635,109022,0.000068,0.605915,408,608,0.995098,0.356044,7
2,0.5,56635,57928,0.000036,0.395745,255,344,0.909804,0.349522,10
3,0.6,56635,28118,0.000018,0.224861,153,173,0.869281,0.596481,12



Summary table saved to:
outputs/tau_sensitivity_summary.csv
